# M4 §4 — Bayesian Team Ratings (PyMC, Bradley–Terry)

**Model.** Strengths $s_i \sim \mathcal{N}(1500, 350^2)$ on the Elo scale;
$P(i	ext{ beats }j) = \sigma\!ig((s_i - s_j)/173ig)$ — why 173: $400/\ln 10 pprox 173.7$,
so σ(ΔR/173) on this scale equals the Elo curve σ(ΔR·ln10/400). Each match enters in
BOTH orientations (symmetrized likelihood) so team1/team2 column order can't matter.

**The Helsing sentence:** the posterior IS the Bayesian update after each match;
Elo is a degenerate online approximation of it (one gradient step per match with a
fixed learning rate K — see `notebooks/elo_derivation.ipynb`).

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from cs2analytics.models.bayes import MIN_SERIES_PER_TEAM, fit_bayesian_ratings, prepare_bt_data

REPO = Path.cwd()
series = pd.read_csv(REPO / "outputs" / "series_clean.csv")

t1_idx, t2_idx, teams = prepare_bt_data(
    series, tier="tier1", year_start=2025, year_end=2025, min_series=MIN_SERIES_PER_TEAM
)
sub = series[
    series["tier"].eq("tier1")
    & pd.to_datetime(series["datetime"], utc=True, format="ISO8601").dt.year.eq(2025)
]
counts = pd.concat([sub["team1"], sub["team2"]]).value_counts()
keep = set(counts[counts >= MIN_SERIES_PER_TEAM].index)
sub = sub[sub["team1"].isin(keep) & sub["team2"].isin(keep)]
result = (sub["winner"] == sub["team1"]).to_numpy().astype(float)
print(f"teams: {len(teams)} | matches: {len(sub)} | team1 win share: {result.mean():.3f}")

teams: 60 | matches: 1160 | team1 win share: 0.553


In [2]:
ratings = fit_bayesian_ratings(t1_idx, t2_idx, result, teams, draws=500, tune=500, chains=2)
ratings.head(10)

Initializing NUTS using jitter+adapt_diag...


Multiprocess sampling (2 chains in 2 jobs)


NUTS: [strength]


Sampling 2 chains for 500 tune and 500 draw iterations (1_000 + 1_000 draws total) took 6 seconds.


We recommend running at least 4 chains for robust computation of convergence diagnostics


The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details


The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details


,team,posterior_mean,hdi_3,hdi_97
0,Team Vitality,1966.447221,1850.835374,2090.275496
1,Team Spirit,1857.284629,1739.717920,1970.526499
2,MOUZ,1803.982440,1687.188528,1911.417335
3,Team Falcons,1757.321821,1643.230598,1868.363974
4,Eternal Fire,1739.188478,1603.154250,1872.507621
5,The Mongolz,1735.432066,1626.293557,1835.257293
6,Betclic,1704.816346,1531.242363,1888.855712
7,Natus Vincere,1703.091584,1591.926494,1810.221729
8,FURIA Esports,1692.988442,1583.296552,1797.620105
9,G2 Esports,1655.827731,1545.625394,1755.518150


## Posterior reading

Top teams get TIGHT intervals (lots of matches → strong likelihood); fringe teams get WIDE
94% HDIs — uncertainty is the point. Compare with Elo's single number: a point estimate
throws away exactly the information a decision-maker needs.

In [3]:
out = REPO / "outputs" / "bayesian_ratings.csv"
ratings.to_csv(out, index=False)
print(f"wrote {out.relative_to(REPO)}: {len(ratings)} teams")

wrote outputs\bayesian_ratings.csv: 60 teams


In [4]:
# Elo comparison (spec §4): posterior-P vs Elo-P on 200 random test pairs
from cs2analytics.features.elo import run_elo_backtest

bt = run_elo_backtest(series, k=32.0)
rng = np.random.default_rng(42)
sample = bt.sample(200, random_state=42)


def elo_p(row):
    e = 1.0 / (1.0 + 10.0 ** ((row["elo_t2_pre"] - row["elo_t1_pre"]) / 400.0))
    return e


def post_p(row):
    s1 = ratings.loc[ratings["team"] == row["team1"], "posterior_mean"]
    s2 = ratings.loc[ratings["team"] == row["team2"], "posterior_mean"]
    if s1.empty or s2.empty:
        return np.nan
    import scipy.special as sp

    return float(sp.expit((s1.iloc[0] - s2.iloc[0]) / 173.0))


cmp = pd.DataFrame(
    {
        "elo_p": sample.apply(elo_p, axis=1),
        "post_p": sample.apply(post_p, axis=1),
    }
).dropna()
mae = (cmp["elo_p"] - cmp["post_p"]).abs().mean()
corr = cmp["elo_p"].corr(cmp["post_p"])
print(f"pairs compared: {len(cmp)} | MAE(elo_p, post_p) = {mae:.4f} | Pearson r = {corr:.3f}")
cmp.describe()

pairs compared: 67 | MAE(elo_p, post_p) = 0.1380 | Pearson r = 0.632


,elo_p,post_p
count,67.000000,67.000000
mean,0.535446,0.525287
std,0.161610,0.229014
min,0.144803,0.073036
25%,0.439395,0.341222
50%,0.539953,0.485700
75%,0.633906,0.696532
max,0.902758,0.967258


## When do Elo and the posterior disagree? (spec §4)

Elo is a *running* estimate — it weighs recent form heavily and never forgets being wrong
about a team's start; its uncertainty is invisible. The posterior pools ALL 2025 tier1
evidence uniformly with a prior — it is stationary (no recency), and it states its own
uncertainty via the HDI width. They disagree most for teams with FEW matches: Elo pins
them at whatever few results imply, while the posterior shrinks them toward the 1500 prior
(wide HDI). That shrinkage is exactly what a hierarchical/Bayesian treatment buys you.